# PreScale: training the pre-scaling forecaster

**Goal.** Predict total CPU demand of a Kubernetes deployment 3 minutes ahead, as P10 / P50 / P90 bounds, and turn the P90 into a replica recommendation. This runs *before* a spike, while HPA only reacts after it.

**Before you run.** Upload to the Colab Files panel: `features.py`, `online.py`, `training_metrics.csv`, `testing_metrics.csv`.

**How the notebook is organised**
1. Setup and data loading
2. Data audit (is the data clean, and are any columns leaking the answer?)
3. Feature code tests (no look-ahead)
4. Dataset, target and time-based split
5. Training
6. Evaluation against baselines, calibration, replica behaviour
7. Failure modes and confidence fallback
8. Quality gate, save, and the train/serve consistency test
9. Export

Every step prints what to look for. If a check fails, the notebook stops, and no model is saved.

## 1. Setup and loading the data

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

DATA_DIR = os.environ.get("PRESCALE_DATA", "/content")   # Colab default
TRAIN_CSV = os.path.join(DATA_DIR, "training_metrics.csv")
TEST_CSV = os.path.join(DATA_DIR, "testing_metrics.csv")
OUT_MODEL = "prescale_model.joblib"

for path in ["features.py", "online.py", TRAIN_CSV, TEST_CSV]:
    assert os.path.exists(path), f"Missing {path}. Upload it to the Colab Files panel."

from features import (HORIZON_MIN, TARGET_UTIL, MIN_REPLICAS, LAGS, WINDOWS, MIN_HISTORY,
                      DYN_COLS, load_metrics, to_grid, make_features, build_xy, pinball)
from online import (build_grid, predict_from_grid, vm_series, MAX_STALENESS_S, LOOKBACK_MIN)

try:
    import lightgbm as lgb
    BACKEND = "lightgbm"
except ImportError:
    from sklearn.ensemble import HistGradientBoostingRegressor
    BACKEND = "sklearn"
print("model backend:", BACKEND)
print("policy: horizon", HORIZON_MIN, "min | target utilisation", TARGET_UTIL,
      "| min replicas", MIN_REPLICAS)

**What to look for:** the backend should say `lightgbm` on Colab. If it says `sklearn`, LightGBM is missing, and results will differ slightly from the reported ones.

In [ ]:
raw_train = load_metrics(TRAIN_CSV)
raw_test = load_metrics(TEST_CSV)
print("train shape:", raw_train.shape, "| test shape:", raw_test.shape)
raw_train.head(3)

## 2. Data audit

Before modelling, check the data itself. A model trained on bad data looks good in testing and fails in production, so each check below is there for a reason.

- **Time coverage and gaps:** missing minutes make lag features wrong.
- **Duplicate timestamps:** the files interleave five deployments, so duplicates must be checked per deployment.
- **Missing values:** the models can't use them.
- **Duplicate columns:** `arrival_rate_rps` should be identical to `request_rate_rps`.
- **Label leakage:** some columns may be computed from the future, which would make the model look better than it is.

In [ ]:
def audit(df, name):
    ts = pd.to_datetime(df["timestamp_utc"], utc=True)
    print(f"== {name}: {len(df)} rows, {df['deployment'].nunique()} deployments")
    print(f"   time range: {ts.min()}  ->  {ts.max()}")
    print(f"   missing values: {int(df.isna().sum().sum())}")
    rows = []
    for dep, g in df.assign(ts=ts).groupby("deployment"):
        g = g.sort_values("ts")
        step = g["ts"].diff().dt.total_seconds().dropna()
        rows.append({
            "deployment": dep,
            "rows": len(g),
            "duplicate_ts": int(g["ts"].duplicated().sum()),
            "gaps_over_1min": int((step > 60).sum()),
            "startup_s": sorted(g["pod_startup_seconds"].unique().tolist()),
            "cpu_request": sorted(g["cpu_request_per_pod_cores"].unique().tolist()),
            "scheduled_min": int(g["scheduled_workload_flag"].sum()),
            "oom_minutes": int((g["oom_kills_1m"] > 0).sum()),
        })
    print(pd.DataFrame(rows).to_string(index=False))

audit(raw_train, "TRAIN")
audit(raw_test, "TEST")

**What to look for:**
- `duplicate_ts` should be 0 for every deployment. A non-zero value means the same minute was recorded twice.
- `gaps_over_1min` should be 0 in training. The test file has one gap, caused by the corrupted row that was skipped; the grid forward-fills gaps of up to 2 minutes.
- `startup_s` is 45–90 s, which is why the horizon is 3 minutes: the forecast has to look far enough ahead to cover a pod starting from scratch.

In [ ]:
# Duplicate columns: are any signals copies of each other?
both = pd.concat([raw_train, raw_test])
print("arrival_rate == request_rate :", np.isclose(both.arrival_rate_rps, both.request_rate_rps).mean().round(4))
print("processing_rate == request_rate:", np.isclose(both.processing_rate_rps, both.request_rate_rps).mean().round(4))
print("ready_pods <= pod_count      :", (both.ready_pods <= both.pod_count).mean().round(4))

**Label leakage check.** `recommended_replicas` is in the CSV. If it is computed from *future* CPU, using it as an input (or measuring against it) would be cheating. This check asks: for each row, does `recommended_replicas` match `ceil(CPU k minutes later / (request x utilisation))`? A high match rate at some `k` means the column is derived from the future, and the notebook will not use it.

In [ ]:
def leakage_check(df):
    out = []
    for dep, g in df.groupby("deployment"):
        g = g.reset_index(drop=True)
        best = (0.0, None, None)
        for k in [0, 1, 2, 3, 5, 8, 10, 15]:
            fut = g["cpu_used_cores"].shift(-k)
            for util in [0.5, 0.6, 0.7, 0.8]:
                pred = np.maximum(1, np.ceil(fut / (g["cpu_request_per_pod_cores"] * util)))
                match = (pred == g["recommended_replicas"]).mean()
                if match > best[0]:
                    best = (match, k, util)
        out.append({"deployment": dep, "best_match": round(best[0], 3),
                    "minutes_ahead": best[1], "util": best[2]})
    return pd.DataFrame(out)

print(leakage_check(raw_train).to_string(index=False))
print("\nInterpretation: matches of 0.5-0.85 at 8-10 minutes ahead mean the column encodes the future.")
print("It is excluded from features and from every metric in this notebook.")

## 3. Feature code tests

The model is only valid if every feature at minute *t* uses information available at *t*. Two tests check that automatically:

1. **No look-ahead:** features computed on the full history must equal features computed on history cut off at *t*, for random *t*. If a feature peeks forward, this fails.
2. **Warm-up:** the first `MIN_HISTORY` minutes have incomplete features and must be dropped.

In [ ]:
g_check = to_grid(raw_train[raw_train["deployment"] == "checkout-api"].sort_values("timestamp_utc"))
X_full = make_features(g_check)

rng = np.random.default_rng(0)
sample_times = X_full.index[MIN_HISTORY + 10:-10][rng.choice(len(X_full) - MIN_HISTORY - 20, 25, replace=False)]
worst = 0.0
for t in sample_times:
    X_cut = make_features(g_check.loc[:t])
    diff = (X_full.loc[t] - X_cut.loc[t]).abs().max()
    worst = max(worst, float(diff))
print(f"look-ahead test over {len(sample_times)} random minutes: max difference = {worst:.2e}")
assert worst < 1e-6, "A feature uses future data. Do not continue."
print("PASS: no feature uses future data.")

**What to look for:** the difference must be essentially zero (floating-point noise). If the assertion fails, training would learn from information the live system doesn't have, and the notebook stops.

## 4. Build the datasets: features, target, and the time-based split

**Target.** We predict `delta = CPU(t+3 min) - CPU(t)` rather than the raw future CPU. Trees cannot extrapolate beyond the range they saw in training. Predicting the change from the current value lets the model handle a level it hasn't seen before. Quantiles are preserved under this shift, so the P10/P50/P90 bounds are unchanged.

**Split.** Data is split strictly by time. Training uses the first 80% of the training file; validation uses the last 20%. Because the target looks 3 minutes ahead, the last 3 minutes before the validation cut-off are removed from training. Otherwise a training label would reveal validation-period information. The test file (a separate week) is never touched until the final evaluation.

In [ ]:
def build_dataset(df):
    parts = []
    for dep, g in df.groupby("deployment"):
        grid = to_grid(g)
        X, y, ev = build_xy(grid)
        d = X.join(y.rename("y")).join(ev)
        d["deployment"] = dep
        parts.append(d)
    return pd.concat(parts).sort_index()

D_train_all = build_dataset(raw_train)
D_test = build_dataset(raw_test)

FEATURE_COLS = [c for c in D_train_all.columns
                if c not in ("y", "cpu_now", "cpu_future", "replicas", "deployment")]
print("number of model inputs:", len(FEATURE_COLS))
print("examples:", FEATURE_COLS[:8], "...")
print("\ntarget (delta CPU, cores) summary:")
print(D_train_all["y"].describe().round(3).to_string())

In [ ]:
ts_unique = D_train_all.index.unique().sort_values()
cut = ts_unique[int(len(ts_unique) * 0.8)]
gap = pd.Timedelta(minutes=HORIZON_MIN)

D_train = D_train_all[D_train_all.index < cut - gap].dropna(subset=FEATURE_COLS + ["y"])
D_val = D_train_all[D_train_all.index >= cut].dropna(subset=FEATURE_COLS + ["y"])
D_test = D_test.dropna(subset=FEATURE_COLS + ["y"])

dropped_for_gap = len(D_train_all[(D_train_all.index >= cut - gap) & (D_train_all.index < cut)])
print(f"train      {len(D_train):>6} rows  {D_train.index.min()} -> {D_train.index.max()}")
print(f"validation {len(D_val):>6} rows  {D_val.index.min()} -> {D_val.index.max()}")
print(f"test       {len(D_test):>6} rows  {D_test.index.min()} -> {D_test.index.max()}  (unseen week)")
print(f"gap rows removed between train and validation: {dropped_for_gap // 5} minutes per deployment")

**What to look for:** the train, validation and test periods must not overlap, and train must end before validation starts. The test period (Oct 15–21) should be a different week from training (Oct 1–7).

## 5. Training: P10, P50, P90 models

Each quantile is a separate model trained with the pinball loss, which is the standard loss for quantile forecasting. Early stopping on the validation set prevents overfitting. The P10 and P90 models set the uncertainty band; P50 is the central forecast.

In [ ]:
QUANTILES = [0.1, 0.5, 0.9]

def make_model(q):
    if BACKEND == "lightgbm":
        return lgb.LGBMRegressor(
            objective="quantile", alpha=q, n_estimators=2000, learning_rate=0.03,
            num_leaves=31, min_child_samples=50, subsample=0.8, subsample_freq=1,
            colsample_bytree=0.8, random_state=42, verbose=-1)
    return HistGradientBoostingRegressor(
        loss="quantile", quantile=q, max_iter=400, learning_rate=0.05,
        max_leaf_nodes=31, min_samples_leaf=50, random_state=42)

models, train_log = {}, {}
for q in QUANTILES:
    t0 = time.time()
    m = make_model(q)
    if BACKEND == "lightgbm":
        m.fit(D_train[FEATURE_COLS], D_train["y"],
              eval_set=[(D_val[FEATURE_COLS], D_val["y"])], eval_metric="quantile",
              callbacks=[lgb.early_stopping(100, verbose=False)])
        train_log[q] = f"best iteration {m.best_iteration_}"
    else:
        m.fit(D_train[FEATURE_COLS], D_train["y"])
        train_log[q] = f"{m.n_iter_} iterations"
    models[q] = m
    print(f"P{int(q*100):<3} trained in {time.time()-t0:5.1f}s | {train_log[q]}")

**What to look for:** the best iteration should be well below the maximum (2000 for LightGBM). If it hits the maximum, the model is still improving and could be trained longer. A very small number (under about 20) suggests the model is overfitting quickly.

In [ ]:
def predict_quantiles(X):
    raw = np.column_stack([models[q].predict(X) for q in QUANTILES])
    return np.sort(raw, axis=1)   # sorting guarantees P10 <= P50 <= P90

val_delta = predict_quantiles(D_val[FEATURE_COLS])
val_now = D_val["cpu_now"].to_numpy()
val_p10, val_p50, val_p90 = (val_now[:, None] + val_delta).T
val_rel = (val_p90 - val_p10) / np.maximum(val_p50, 0.05)
MAX_REL_WIDTH = float(np.quantile(val_rel, 0.90))
print(f"confidence threshold (90th percentile of validation relative width): {MAX_REL_WIDTH:.3f}")
print("meaning: about 10% of minutes will be too uncertain and handed to HPA")

## 6. Evaluation

Judge the model against simple baselines, not against zero:
- **Persistence:** "the CPU in 3 minutes will equal the CPU now." A model that cannot beat this is not useful.
- **Reactive HPA:** provision for the CPU *now*. This is what the system does today.

Report both the forecast accuracy and the operational outcome: did the replicas cover the demand when it arrived?

In [ ]:
def evaluate(D, label):
    pred_delta = predict_quantiles(D[FEATURE_COLS])
    now = D["cpu_now"].to_numpy()
    y = D["y"].to_numpy()
    p10, p50, p90 = (now[:, None] + pred_delta).T
    per_pod = D["cpu_request"].to_numpy() * TARGET_UTIL
    future = D["cpu_future"].to_numpy()

    need = np.maximum(MIN_REPLICAS, np.ceil(future / per_pod))            # what was actually needed
    rec_prescale = np.maximum(MIN_REPLICAS, np.ceil(p90 / per_pod))       # PreScale recommendation
    rec_hpa = np.maximum(MIN_REPLICAS, np.ceil(now / per_pod))            # reactive HPA

    res = {
        "minutes": len(D),
        "MAE P50 (model)": np.mean(np.abs(y - (p50 - now))),
        "MAE persistence": np.mean(np.abs(y)),
        "P10-P90 coverage (target 0.80)": np.mean((future >= p10) & (future <= p90)),
        "under-provisioned: PreScale": np.mean(rec_prescale < need),
        "under-provisioned: HPA": np.mean(rec_hpa < need),
        "extra pods: PreScale": np.mean(np.maximum(0, rec_prescale - need)),
        "extra pods: HPA": np.mean(np.maximum(0, rec_hpa - need)),
    }
    print(f"== {label}")
    for k, v in res.items():
        print(f"   {k:34s} {v:.4f}" if isinstance(v, float) else f"   {k:34s} {v}")
    return res, pd.DataFrame(
        {"dep": D["deployment"].to_numpy(), "err_m": np.abs(y - (p50 - now)),
         "err_p": np.abs(y), "cov": (future >= p10) & (future <= p90),
         "u_p": rec_prescale < need, "u_h": rec_hpa < need}
    ).groupby("dep").agg(MAE_model=("err_m", "mean"), MAE_persist=("err_p", "mean"),
                         coverage=("cov", "mean"), underprov_PreScale=("u_p", "mean"),
                         underprov_HPA=("u_h", "mean")).round(3)

val_res, _ = evaluate(D_val, "VALIDATION (last 20% of training window)")
test_res, test_by_dep = evaluate(D_test, "TEST (Oct 15-21, unseen week)")
print("\nPer deployment on TEST:")
print(test_by_dep.to_string())

**What to look for:**
- **MAE P50 below MAE persistence:** the model adds value. The gap is modest; a large improvement would be surprising for a 3-minute horizon.
- **Coverage near 0.80:** the P10–P90 band should contain about 80% of outcomes. Much lower means the band is too narrow and the model is overconfident.
- **Under-provisioned PreScale below HPA:** this is the reason the system exists.
- **Extra pods:** PreScale will usually provision more. Check whether that cost is acceptable for the SLA.

### 6b. Calibration: are the quantiles honest?

If the P10 is right, the actual value should fall below it about 10% of the time. If the P50 is right, it should be below about 50%. This is a direct test of whether the uncertainty estimates mean what they say.

In [ ]:
def calibration(D):
    pred = predict_quantiles(D[FEATURE_COLS])
    y = D["y"].to_numpy()
    return {q: float(np.mean(y <= pred[:, i])) for i, q in enumerate(QUANTILES)}

cal_test = calibration(D_test)
print("share of actual outcomes below each predicted quantile (TEST):")
for q, v in cal_test.items():
    print(f"   P{int(q*100)}: observed {v:.3f}   expected {q:.2f}")

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot([0, 1], [0, 1], "k--", label="perfect")
ax.plot(list(QUANTILES), [cal_test[q] for q in QUANTILES], "o-", label="PreScale (test)")
ax.set_xlabel("expected share below quantile"); ax.set_ylabel("observed share below quantile")
ax.set_title("Calibration (closer to diagonal is better)"); ax.legend(); ax.grid(alpha=.3)
plt.show()

**What to look for:** points near the dashed diagonal. The P90 observed value is typically a little below 0.90 here, which means the upper bound is slightly too low: spikes above P90 happen somewhat more often than the model expects. This is a known limitation worth stating when you report results.

## 6c. Visual check: forecast against reality

In [ ]:
dep_plot = "checkout-api"
day = D_test[(D_test["deployment"] == dep_plot) &
             (D_test.index >= "2026-10-18") & (D_test.index < "2026-10-19")]
pred = predict_quantiles(day[FEATURE_COLS])
now = day["cpu_now"].to_numpy()
target_time = day.index + pd.Timedelta(minutes=HORIZON_MIN)   # forecast is about this future minute

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(target_time, day["cpu_future"], color="black", lw=1, label="actual CPU")
ax.fill_between(target_time, now + pred[:, 0], now + pred[:, 2], color="tab:blue", alpha=.25, label="P10-P90")
ax.plot(target_time, now + pred[:, 1], color="tab:blue", lw=1, label="P50 forecast")
ax.set_title(f"{dep_plot}: forecast made 3 minutes earlier vs actual, 18 Oct (test week)")
ax.set_ylabel("CPU cores"); ax.legend(loc="upper right"); ax.grid(alpha=.3)
plt.show()

**What to look for:** the actual line should sit inside the shaded band most of the time, and the P50 line should follow the rise and fall of the load. Big misses that line up with sudden spikes are the cases to watch for; a pure forecast cannot predict a spike that has no warning in the recent history.

## 6d. Replica behaviour over a day

In [ ]:
need_day = np.maximum(MIN_REPLICAS, np.ceil(day["cpu_future"] / (day["cpu_request"] * TARGET_UTIL)))
rec_p_day = np.maximum(MIN_REPLICAS, np.ceil((now + pred[:, 2]) / (day["cpu_request"] * TARGET_UTIL)))
rec_h_day = np.maximum(MIN_REPLICAS, np.ceil(now / (day["cpu_request"] * TARGET_UTIL)))

fig, ax = plt.subplots(figsize=(13, 4))
ax.step(target_time, need_day, where="post", color="black", lw=1.5, label="needed (hindsight)")
ax.step(target_time, rec_p_day, where="post", color="tab:blue", lw=1, label="PreScale recommendation")
ax.step(target_time, rec_h_day, where="post", color="tab:orange", lw=1, label="reactive HPA")
ax.set_ylabel("replicas"); ax.set_title(f"{dep_plot}: replicas over 18 Oct"); ax.legend(); ax.grid(alpha=.3)
plt.show()

**What to look for:** the blue (PreScale) line should rise before the black "needed" line during spikes, while the orange (HPA) line rises after. Where blue is above black for long stretches, that is the extra capacity cost.

## 7. Failure modes: when the model should refuse to decide

The most important property for a production model is knowing when not to act. Three cases:

1. **Low confidence:** the forecast band is too wide. These minutes fall back to HPA. We check that the fallback minutes really are the hard ones (their error should be higher).
2. **Stale data:** metrics stop arriving. The model must not act on old information.
3. **Missing data:** a gap longer than 2 minutes produces undefined features, so the model must not guess.

In [ ]:
test_now = D_test["cpu_now"].to_numpy()
test_pred = predict_quantiles(D_test[FEATURE_COLS])
t10, t50, t90 = (test_now[:, None] + test_pred).T
fallback_mask = (t90 - t10) / np.maximum(t50, 0.05) > MAX_REL_WIDTH
y_test = D_test["y"].to_numpy()
err = np.abs(y_test - (t50 - test_now))

print(f"minutes handed to HPA: {fallback_mask.mean():.1%}")
print(f"MAE on confident minutes : {err[~fallback_mask].mean():.4f}")
print(f"MAE on fallback minutes  : {err[fallback_mask].mean():.4f}")
print("Fallback minutes are harder, as intended." if err[fallback_mask].mean() > err[~fallback_mask].mean()
      else "WARNING: fallback minutes are not harder; review MAX_REL_WIDTH.")

In [ ]:
# Explicit checks of the three failure modes, using the same function the service uses
bundle_for_tests = {"models": models, "quantiles": QUANTILES, "feature_cols": FEATURE_COLS,
                    "max_rel_width": MAX_REL_WIDTH, "target_util": TARGET_UTIL,
                    "min_replicas": MIN_REPLICAS, "horizon_min": HORIZON_MIN}
some_t = D_test.index[1000]
base_grid = to_grid(raw_test[raw_test["deployment"] == "checkout-api"])
now_t = some_t

short = base_grid.loc[:now_t].iloc[-10:]                     # too little history
print("short history  ->", predict_from_grid(short, bundle_for_tests, now_t)["reason"])

stale_now = now_t + pd.Timedelta(seconds=MAX_STALENESS_S + 300)
print("stale data     ->", predict_from_grid(base_grid.loc[:now_t], bundle_for_tests, stale_now)["reason"])

gapped = base_grid.loc[:now_t].copy()
gapped.iloc[-5:-1, gapped.columns.get_loc("cpu")] = np.nan   # 4-minute hole in the recent history
print("gap in history ->", predict_from_grid(gapped, bundle_for_tests, now_t)["reason"])

**What to look for:** each case returns `fallback_hpa` with a clear reason. Nothing should return `prescale` in these cases.

## 8. Quality gate, save, and the train/serve consistency test

**Gate.** The model is saved only if it beats persistence, keeps coverage between 70% and 90%, and under-provisions less than reactive HPA on the unseen week. If any check fails, nothing is saved.

**Consistency test.** Training uses the full history. Production uses a 60-minute window pulled from VictoriaMetrics. This test replays real test-week data through the production path and checks that the answers match. This is the test that catches silent train/serve skew.

In [ ]:
checks = {
    "beats persistence (MAE)": test_res["MAE P50 (model)"] < test_res["MAE persistence"],
    "coverage in [0.70, 0.90]": 0.70 <= test_res["P10-P90 coverage (target 0.80)"] <= 0.90,
    "fewer under-provisioned minutes than HPA":
        test_res["under-provisioned: PreScale"] < test_res["under-provisioned: HPA"],
}
for name, ok in checks.items():
    print(("PASS  " if ok else "FAIL  ") + name)
GATE_PASSED = all(checks.values())
print("GATE:", "PASSED" if GATE_PASSED else "FAILED: model will not be saved")

In [ ]:
def vm_payload(sub, col):
    # Mimic a VictoriaMetrics query_range response for one signal
    vals = [[str(int(t.timestamp())), str(v)] for t, v in zip(sub["ts"], sub[col])]
    return {"status": "success", "data": {"resultType": "matrix", "result": [{"metric": {}, "values": vals}]}}

static_cfg = raw_train.groupby("deployment").agg(
    cpu_request=("cpu_request_per_pod_cores", "first"),
    mem_limit=("mem_limit_per_pod_bytes", "first")).to_dict(orient="index")

summary = {}
bundle_tmp = {"models": models, "quantiles": QUANTILES, "feature_cols": FEATURE_COLS,
              "max_rel_width": MAX_REL_WIDTH, "target_util": TARGET_UTIL,
              "min_replicas": MIN_REPLICAS, "horizon_min": HORIZON_MIN}

for dep in sorted(raw_test["deployment"].unique()):
    g = raw_test[raw_test["deployment"] == dep].copy()
    g["ts"] = pd.to_datetime(g["timestamp_utc"], utc=True).dt.tz_convert(None)
    offline_grid = to_grid(g.sort_values("ts"))
    offline_X = make_features(offline_grid)
    pool = offline_X.index[(offline_X.index >= "2026-10-16") & (offline_X.index < "2026-10-21")]
    picks = np.random.default_rng(7).choice(len(pool), size=60, replace=False)

    worst, compared, fallbacks = 0.0, 0, 0
    for p in picks:
        t = pool[p]
        window = g[(g["ts"] >= t - pd.Timedelta(minutes=LOOKBACK_MIN)) & (g["ts"] <= t)]
        series = {
            "cpu": vm_series(vm_payload(window, "cpu_used_cores")),
            "rps": vm_series(vm_payload(window, "request_rate_rps")),
            "mem": vm_series(vm_payload(window, "mem_used_bytes")),
            "pod_count": vm_series(vm_payload(window, "pod_count")),
            "ready_pods": vm_series(vm_payload(window, "ready_pods")),
        }
        result = predict_from_grid(build_grid(series, static_cfg[dep]), bundle_tmp, now=t)
        if result["mode"] != "prescale":
            fallbacks += 1
            continue
        x = offline_X.loc[[t], FEATURE_COLS]
        offline_q = np.sort([models[q].predict(x)[0] for q in QUANTILES]) + offline_grid.loc[t, "cpu"]
        diff = max(abs(result["cpu_forecast_cores"][k] - round(v, 4))
                   for k, v in zip(["p10", "p50", "p90"], offline_q))
        worst = max(worst, diff)
        compared += 1
    summary[dep] = {"compared": compared, "fallbacks": fallbacks, "max_difference": worst}

consistency = pd.DataFrame(summary).T
print(consistency.to_string())
assert (consistency["max_difference"] <= 5e-5).all(), "Online and offline predictions disagree. Do not deploy."
print("\nPASS: online path reproduces offline predictions for every checked minute.")

**What to look for:** `max_difference` should be 0 or below 0.0001. A larger value means the production path computes features differently from training, so the model's answers online would not match what was tested. Fix that before deploying.

## 9. Export

In [ ]:
if not GATE_PASSED:
    raise SystemExit("Gate failed: not saving. Review sections 6 and 8.")

static_cfg = raw_train.groupby("deployment").agg(
    cpu_request=("cpu_request_per_pod_cores", "first"),
    mem_limit=("mem_limit_per_pod_bytes", "first")).to_dict(orient="index")

bundle = {
    "schema_version": 1,
    "backend": BACKEND,
    "models": models,
    "quantiles": QUANTILES,
    "feature_cols": FEATURE_COLS,
    "horizon_min": HORIZON_MIN,
    "target_util": TARGET_UTIL,
    "min_replicas": MIN_REPLICAS,
    "max_rel_width": MAX_REL_WIDTH,
    "deployments": static_cfg,
    "metrics_test": test_res,
    "metrics_validation": val_res,
    "trained_on": "training_metrics.csv (2026-10-01..07)",
    "tested_on": "testing_metrics.csv (2026-10-15..21)",
}
joblib.dump(bundle, OUT_MODEL)

# Reload and check that the saved file gives the same answer
reloaded = joblib.load(OUT_MODEL)
probe = D_test[FEATURE_COLS].iloc[[500]]
same = np.allclose([reloaded["models"][q].predict(probe)[0] for q in QUANTILES],
                   [models[q].predict(probe)[0] for q in QUANTILES])
print("saved:", OUT_MODEL, "| reload check:", "OK" if same else "MISMATCH")

## Before going live

- Confirm the PromQL in `serve_app.py` measures the same quantities as the CSV columns.
- Run the consistency test against the live VictoriaMetrics data, not just the CSV.
- Set up a periodic retrain and a drift alarm. Watch the under-provisioned rate and coverage weekly.
- The model recommends replicas only. OOM-risk and memory-limit decisions are separate components.
- Known gap: the spike-prediction power is limited to what appears in the last hour of metrics. Scheduled batch jobs are not visible to the model.